### Importing Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

### Importing Data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground Season 3 Episode 12/train.csv').set_index('id')
df

### Feature Visualizations

In [ ]:
features = ['gravity', 'ph', 'osmo', 'cond', 'urea', 'calc']
target = 'target'

fig, axes = plt.subplots(nrows=len(features), figsize=(3, 2 * len(features)))

from sklearn.preprocessing import MinMaxScaler
X = df.loc[:, features]
minmax_scaler = MinMaxScaler()
X.loc[:, features] = minmax_scaler.fit_transform(X[features])

print(X)

for i, feature in enumerate(features):
    sns.kdeplot(x=(X[feature]*2)-1, hue=df[target], fill=True, ax=axes[i])
    axes[i].set_title(feature + ' Distribution by Target')
    axes[i].set_xlim(-1.5, 1.5)

plt.tight_layout()
plt.show()

### Feature Engineering

In [ ]:

df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground Season 3 Episode 12/train.csv').set_index('id')
features = ['gravity', 'ph', 'osmo', 'cond', 'urea', 'calc']

X = df.loc[:, features]
minmax_scaler = MinMaxScaler()
X.loc[:, features] = minmax_scaler.fit_transform(X[features])


for i in range(len(features)):
  for j in range(i+1, len(features)):
        ratio_column_name = features[i] + '_' + features[j] + '_ratio'
        diff_column_name = features[i] + '_' + features[j] + '_diff'

        df[ratio_column_name] = X[features[i]] / X[features[j]]
        df[diff_column_name] = X[features[i]] - X[features[j]]

for feature in df.drop(['target'], axis=1).columns:
  feature_mean = df[feature].mean()
  feature_max = df[feature].max()
  feature_min = df[feature].min()

  df['mean_diff_'+feature] = df[feature] - feature_mean
  df['max_diff_'+feature] = df[feature] - feature_max
  df['min_diff_'+feature] = df[feature] - feature_min

for feature in df.drop(['target'], axis=1).columns:
  df['log_'+feature] = np.log(df[feature])
  df['sqrt_'+feature] = np.sqrt(df[feature])

df = df.replace([np.inf, -np.inf], np.nan)
for col in df.columns:
    col_mean = df[col].mean()
    df.loc[:, col] = df[col].fillna(col_mean)

df = df.dropna(axis=1)
df.isna().sum()

### Feature Selection

In [ ]:

df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground Season 3 Episode 12/train.csv').set_index('id')
features = ['gravity', 'ph', 'osmo', 'cond', 'urea', 'calc']

df['sqrt_calc'] = np.sqrt(df['calc'])
df['log_min_diff_calc'] = np.log(df['calc']-df['calc'].min())
df['log_min_diff_calc'].replace([np.inf, -np.inf], 0, inplace=True)

df['sqrt_log_calc'] = np.sqrt(np.log(df['calc']))
df.loc[np.log(df['calc']) < 0, 'sqrt_log_calc'] = np.sqrt(-np.log(df['calc']))


np.isinf(df).sum()
df.isna().sum()
df.corr()['target'].abs().sort_values(ascending=False)[:40]

### Preparing Data

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

X = df.drop(['target'], axis=1)
X = minmax_scaler.fit_transform(X)

y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

X_train.shape, X_test.shape, y_train.shape, y_test.shape
df.drop(['target'], axis=1).columns                                 

### Model Selection

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, SGDRegressor
from sklearn.metrics import roc_auc_score

regressors = [
    LinearRegression(),
    Ridge(),
    Lasso(),
    ElasticNet(),
    BayesianRidge(),
    SGDRegressor()
]

for reg in regressors:
    reg.fit(X_train, y_train)
    predictions = reg.predict(X_test)
    positive_auc = roc_auc_score(y_test, predictions)
    name = reg.__class__.__name__
    print(name, positive_auc)



### Visualizing Results

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, recall_score, precision_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LinearRegression

model = Ridge()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
positive_auc = roc_auc_score(y_test, predictions)

positive_fpr, positive_tpr, _ = roc_curve(y_test, predictions)
plt.scatter(positive_fpr, positive_tpr, marker='.', label='Logistic')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

print('Logistic: %.3f' % (positive_auc))

### Model Tuning

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold

# Create a linear regression model
model = Ridge()

# Define the hyperparameters to tune
params = {
    'alpha': [0.01, 0.1, 0.5, 1.0],
    'fit_intercept': [True, False],
    'tol': [1e1-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6],
    'solver': ['auto', 'svd', 'lsqr', 'sparse_cg', 'sag', 'saga'],
    'max_iter': [2000, 5000],
    'random_state': [0, 42, 123],
    'copy_X': [True, False],
    'positive': [True, False],
}

# Create a grid search object
model = GridSearchCV(estimator=model, param_grid=params, 
                            cv=KFold(n_splits=5, shuffle=True), 
                            scoring='neg_mean_squared_error', verbose=3)

# Fit the grid search object to the training data
model.fit(X_train, y_train)
best_params = model.best_params_

# Print the best score and parameters
print(model.best_score_)
print(model.best_params_)


In [ ]:
model = Ridge(**best_params)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
positive_auc = roc_auc_score(y_test, predictions)

positive_fpr, positive_tpr, _ = roc_curve(y_test, predictions)
plt.scatter(positive_fpr, positive_tpr, marker='.', label='Logistic')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

print('Logistic: %.3f' % (positive_auc))

### Final Submission

In [ ]:
submission = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground Season 3 Episode 12/test.csv').loc[:,['id']]
final_X = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground Season 3 Episode 12/test.csv').set_index('id')
final_X.head()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import FunctionTransformer

final_model = LinearRegression()

def create_features(X):
    X['sqrt_calc'] = np.sqrt(X['calc'])
    X['log_min_diff_calc'] = np.log(X['calc']-X['calc'].min())
    X['log_min_diff_calc'].replace([np.inf, -np.inf], 0, inplace=True)
    X['sqrt_log_calc'] = np.sqrt(np.log(X['calc']))
    X['sqrt_log_calc'].fillna(value=X['sqrt_log_calc'].mean(), inplace=True)
    return X

def select_features(X):
  return X[['gravity', 'ph', 'osmo', 'cond', 'urea', 'calc', 'sqrt_calc', 'log_min_diff_calc', 'sqrt_log_calc']]

pipeline = Pipeline([
    ('create_features', FunctionTransformer(create_features)),
    ('select_features', FunctionTransformer(select_features)),
    ('minmax_scaler', MinMaxScaler()),
    ('model', final_model)
])


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground Season 3 Episode 12/train.csv').set_index('id')

X = df.drop(['target'], axis=1)
y = df['target']
pipeline.fit(X, y)

submission['target'] = pipeline.predict(final_X)

In [ ]:

submission.to_csv('/content/drive/MyDrive/Kaggle Competitions/Playground Season 3 Episode 12/submission.csv', index=False)
pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Playground Season 3 Episode 12/submission.csv')
